<a href="https://colab.research.google.com/github/SSK-KGP/ReLeaf/blob/main/To_Tflite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
BASE_PATH = '/content/drive/MyDrive/PlantVillage/'
os.makedirs(BASE_PATH, exist_ok=True)

Mounted at /content/drive


Importing libraries, defining hyperparameters and loading the model

In [ ]:
import os, json
import numpy as np
import tensorflow as tf

KERAS_MODEL = BASE_PATH + "best_plant_disease_cnn.keras"
LABELS_FILE = BASE_PATH + "class_names.json"
TFLITE_OUT = BASE_PATH + "plant_disease_model.tflite"
TFLITE_QUANT = BASE_PATH + "plant_disease_model_quant.tflite"
IMG_HEIGHT = 224
IMG_WIDTH = 224

model = tf.keras.models.load_model(KERAS_MODEL)

Convert to standard TFLite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(TFLITE_OUT, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_OUT) / 1e6
print(f"Size = {size_mb:.1f} MB")


Saved artifact at '/tmp/tmpepgeupqo'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  136400678641616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678643920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678644112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678642192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678644688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678643344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678646032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678645840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678645264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678644496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678646

Convert to quantized Tflite

In [ ]:
converter_q = tf.lite.TFLiteConverter.from_keras_model(model)
converter_q.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant = converter_q.convert()

with open(TFLITE_QUANT, "wb") as f:
    f.write(tflite_quant)

size_mb_q = os.path.getsize(TFLITE_QUANT) / 1e6
print(f"Size = {size_mb_q:.1f} MB and {(size_mb/size_mb_q):.1f}x smaller")

Saved artifact at '/tmp/tmp30dy0bfx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  136400678641616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678643920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678644112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678642192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678644688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678643344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678646032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678645840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678645264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678644496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136400678646

Sanity Check

In [ ]:
!pip install ai-edge-litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 61.5 MB/s eta 0:00:00


In [ ]:
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path = TFLITE_QUANT)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

dummy_input = np.random.rand(1, IMG_HEIGHT, IMG_WIDTH, 3).astype(np.float32)
interpreter.set_tensor(input_details[0]["index"], dummy_input)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]["index"])

with open(LABELS_FILE) as f:
    class_names = json.load(f)

top_idx = np.argsort(output[0])[-5:][::-1]
print(dummy_input)
print(f"Input Shape: {input_details[0]['shape']}")
print(f"Output Shape: {output_details[0]['shape']}")
for i in top_idx:
    print(f"{class_names[i]}: {output[0][i]}")

[[[[0.9826212  0.5202724  0.46469754]
   [0.89003897 0.12956452 0.78974867]
   [0.9953182  0.50388104 0.4164902 ]
   ...
   [0.9379653  0.3990126  0.928547  ]
   [0.74542457 0.44893956 0.9575352 ]
   [0.8167205  0.30259696 0.9908686 ]]

  [[0.79297537 0.8802871  0.95304525]
   [0.29696155 0.21613899 0.43573242]
   [0.8385424  0.14954911 0.67375165]
   ...
   [0.51808816 0.38614365 0.5733624 ]
   [0.57102436 0.01614428 0.08495965]
   [0.46805477 0.6120649  0.13382953]]

  [[0.11419752 0.56283414 0.04643091]
   [0.7266053  0.79591286 0.44186252]
   [0.7715807  0.0111819  0.47204316]
   ...
   [0.9153855  0.77927035 0.54811656]
   [0.79467285 0.7145009  0.5558741 ]
   [0.6891854  0.7113228  0.09118826]]

  ...

  [[0.7092257  0.96213615 0.04133797]
   [0.22036204 0.57925767 0.91765773]
   [0.3398097  0.32308418 0.63875234]
   ...
   [0.7195259  0.62098885 0.83721864]
   [0.7653507  0.3803278  0.9530492 ]
   [0.03446001 0.65569466 0.37299374]]

  [[0.39934233 0.9304932  0.7243626 ]
   [0.0